# 1. Clone & Setup

**IMPORTANT NOTE:** Make sure to check the cloned branch on `computer-vision-underwater-image-enhancer` repositories.

In [ ]:
%%bash
R=/kaggle/working/computer-vision-underwater-image-enhancer
[ -d $R ] || git clone -q -b feature/benchmark-batch-enhance-ruod --single-branch https://github.com/bao-nguyen-quoc/computer-vision-underwater-image-enhancer $R
git -C $R pull -q
bash $R/enhancers/setup.sh

# 2. Optional verification

## 1. Enhance samples by 2 model (Semi-URI & SCNet)

In [ ]:
%%bash
R=/kaggle/working/computer-vision-underwater-image-enhancer
python $R/enhancers/enhance_semiuir.py --repo /kaggle/working/computer-vision-Semi-UIR --input $R/samples/test_img.png --output /kaggle/working/out_semiuir
python $R/enhancers/enhance_scnet.py --repo /kaggle/working/computer-vision-SCNet --input $R/samples/test_img.png --output /kaggle/working/out_scnet

## 2. Verify after enhancement

In [ ]:
import numpy as np
from PIL import Image
R = '/kaggle/working/computer-vision-underwater-image-enhancer'
!python {R}/enhancers/check_outputs.py --orig {R}/samples --out /kaggle/working/out_semiuir /kaggle/working/out_scnet
paths = [f'{R}/samples/test_img.png', '/kaggle/working/out_semiuir/test_img.png', '/kaggle/working/out_scnet/test_img.png']
display(Image.fromarray(np.hstack([np.asarray(Image.open(p).convert('RGB')) for p in paths])))   # Origin | Semi-UIR | SCNet

# 3. PSNR / SSIM on test dataset of UIED

## 3.1. Enhance on UIEB test

In [ ]:
import glob, os, subprocess

# Mục 3 là tuỳ chọn (PSNR/SSIM trên UIEB). Không thấy dữ liệu thì bỏ qua, không làm dừng notebook.
found = glob.glob("/kaggle/input/**/UIEBD/test", recursive=True)
UIEB = found[0] if found else None
print("UIEB test:", UIEB)
if UIEB:
    print(os.listdir(UIEB)) # Expected: ['image', 'label']
else:
    print("Không tìm thấy UIEBD/test -> bỏ qua mục 3. Cấu trúc input hiện có:")
    print(subprocess.run("find /kaggle/input -maxdepth 6 -type d | head -40", shell=True, capture_output=True, text=True).stdout)

In [ ]:
%%bash
set -e
D=$(find /kaggle/input -type d -path '*/UIEBD/test' -print -quit)
if [ -z "$D" ]; then echo "Bỏ qua mục 3.1: không có UIEBD/test"; exit 0; fi
IN=$D/image
OUT=/kaggle/working/out_benchmark
E=/kaggle/working/computer-vision-underwater-image-enhancer/enhancers

python $E/enhance_scnet.py --repo /kaggle/working/computer-vision-SCNet --input $IN --output $OUT/uieb_scnet
python $E/enhance_semiuir.py --repo /kaggle/working/computer-vision-Semi-UIR --input $IN --output $OUT/uieb_semiuir
python $E/check_outputs.py --orig $IN --out $OUT/uieb_semiuir $OUT/uieb_scnet

## 3.2. Calculate PSNR/SSIM

In [ ]:
import glob, subprocess
found = glob.glob("/kaggle/input/**/UIEBD/test", recursive=True)
if not found:
    print("Bỏ qua mục 3.2: không có UIEBD/test")
else:
    D   = found[0]
    IN  = f"{D}/image"
    OUT = "/kaggle/working/out_benchmark"
    E   = "/kaggle/working/computer-vision-underwater-image-enhancer/enhancers"
    r = subprocess.run(["python", f"{E}/eval_uieb.py", "--ref", f"{D}/label",
                        "--pred", f"raw={IN}", f"scnet={OUT}/uieb_scnet", f"semiuir={OUT}/uieb_semiuir"],
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-1500:])

# 4. Enhance RUOD

In [ ]:
# Optional: Check current path and GPU config
!find /kaggle/input -type d -path '*/orig/images/*'
!nvidia-smi -L

## 4.1. Test with 100 images for time and VRAM estimation

In [ ]:
import os, re, glob, shutil, subprocess
# Tự tìm thư mục gốc ruod640 (chứa orig/images/train)
found = glob.glob("/kaggle/input/**/orig/images/train", recursive=True)
assert found, "Không tìm thấy ruod640/orig/images/train trong input -> kiểm tra lại dataset ruod640"
R640 = os.path.dirname(os.path.dirname(os.path.dirname(found[0])))
print("ruod640:", R640)
E    = "/kaggle/working/computer-vision-underwater-image-enhancer/enhancers"
P    = "/kaggle/working/probe"; os.makedirs(P, exist_ok=True)
for n in sorted(os.listdir(f"{R640}/orig/images/train"))[:100]:
    if not os.path.exists(f"{P}/{n}"): os.symlink(f"{R640}/orig/images/train/{n}", f"{P}/{n}")

# theo dõi VRAM ở nền (Popen được phép chạy nền)
mon = subprocess.Popen("nvidia-smi --query-gpu=index,memory.used --format=csv,noheader,nounits -l 1 > /kaggle/working/vram.log", shell=True)

def probe(script, repo, out):
    r = subprocess.run(["python", f"{E}/{script}", "--repo", repo, "--input", P, "--output", out],
                       capture_output=True, text=True)
    t = [float(x) for x in re.findall(r": ([\d.]+)s$", r.stdout, flags=re.M)]
    print(f"{script}: exit {r.returncode} | {len(t)} ảnh | trung bình (bỏ ảnh đầu): {sum(t[1:]) / max(len(t) - 1, 1):.2f} s/ảnh")
    if r.returncode: print(r.stderr[-600:])

probe("enhance_semiuir.py", "/kaggle/working/computer-vision-Semi-UIR", "/kaggle/working/probe_semiuir")
probe("enhance_scnet.py",   "/kaggle/working/computer-vision-SCNet",     "/kaggle/working/probe_scnet")

mon.terminate(); subprocess.run(["pkill", "-f", "nvidia-smi --query"])
peak = {}
for line in open("/kaggle/working/vram.log"):
    i, v = line.split(","); peak[int(i)] = max(peak.get(int(i), 0), int(v))
print("VRAM đỉnh (MiB) theo GPU:", peak)

# Dọn file thử để output cuối cùng gọn (symlink + ảnh thử không cần giữ)
for d in [P, "/kaggle/working/probe_semiuir", "/kaggle/working/probe_scnet"]:
    shutil.rmtree(d, ignore_errors=True)

## 4.2. Run

In [ ]:
import os, subprocess, time
RUN = "/kaggle/working/computer-vision-underwater-image-enhancer/enhancers/run_ruod.sh"
SCRIPT = {"scnet": "enhance_scnet.py", "semiuir": "enhance_semiuir.py"}

def launch(enhancer, splits):
    if subprocess.run(["pgrep", "-f", SCRIPT[enhancer]], capture_output=True).returncode == 0:
        raise SystemExit(f"{enhancer} is executing.")
    logfile = f"/kaggle/working/run_{enhancer}.log"
    proc = subprocess.Popen(["bash", RUN], cwd="/kaggle/working", stdout=open(logfile, "w"),
                            stderr=subprocess.STDOUT, start_new_session=True,
                            env={**os.environ, "SPLITS": splits, "ENHANCERS": enhancer})
    time.sleep(5)
    print(open(logfile).read())
    return proc

def status(enhancer):
    for s in ["train", "test"]:
        d = f"/kaggle/working/enhanced/{enhancer}/images/{s}"
        print(enhancer, s, len(os.listdir(d)) if os.path.isdir(d) else 0)
    print(subprocess.run(["tail", "-n", "2", f"/kaggle/working/{enhancer}.log"], capture_output=True, text=True).stdout)

Launch SCNet (GPU 0) và Semi-UIR (GPU 1) chạy song song

In [ ]:
proc_scnet = launch("scnet", "train test")

In [ ]:
proc_semiuir = launch("semiuir", "train test")

## 4.3. Chờ cả 2 enhancer chạy xong

**Bắt buộc khi chạy bằng Save Version**: nếu không chờ, notebook kết thúc ngay và tiến trình enhance chạy nền bị tắt giữa chừng. Khi chạy tay bằng ▶ thì cell này cũng dùng để theo dõi: nó in tiến độ mỗi 5 phút.

In [ ]:
import time
procs = {"scnet": proc_scnet, "semiuir": proc_semiuir}
while any(p.poll() is None for p in procs.values()):
    time.sleep(300)
    for e in procs:
        status(e)

rc = {e: p.returncode for e, p in procs.items()}
print("Return codes:", rc)
for e in procs:
    status(e)
    print(f"----- run_{e}.log (cuối) -----")
    print(open(f"/kaggle/working/run_{e}.log").read()[-1500:])   # phải có "ALL OK"
assert all(v == 0 for v in rc.values()), "Có enhancer bị lỗi, xem log ở trên" 

## 4.4. Đóng gói output

Nén mỗi enhancer thành 1 file zip (ảnh + nhãn, train + test), kiểm tra đủ số file rồi mới xoá thư mục gốc. Output cuối gồm `enhanced_scnet.zip`, `enhanced_semiuir.zip` và các file log/kết quả nhỏ, tránh để hơn 56.000 file lẻ trong output.

Notebook khác dùng lại: **Add Input → notebook này**, rồi `unzip -q .../enhanced_scnet.zip -d /kaggle/working/data`.

In [ ]:
import os, shutil, zipfile
ROOT = "/kaggle/working/enhanced"
EXPECT = {"train": 9800, "test": 4200}

for e in ["scnet", "semiuir"]:
    out = f"/kaggle/working/enhanced_{e}.zip"
    n = 0
    with zipfile.ZipFile(out, "w", zipfile.ZIP_STORED) as z:   # ảnh JPEG đã nén sẵn, STORED cho nhanh
        for kind in ["images", "labels"]:
            for s in ["train", "test"]:
                d = f"{ROOT}/{e}/{kind}/{s}"
                names = sorted(os.listdir(d))
                assert len(names) == EXPECT[s], f"{d}: {len(names)} file, cần {EXPECT[s]}"
                for name in names:
                    z.write(f"{d}/{name}", arcname=f"{e}/{kind}/{s}/{name}")
                    n += 1
    with zipfile.ZipFile(out) as z:
        assert len(z.namelist()) == n == 2 * sum(EXPECT.values()), "zip thiếu file"
        assert z.testzip() is None, "zip bị lỗi"
    print(out, n, "file,", round(os.path.getsize(out) / 2**20), "MiB")

# Chỉ xoá sau khi cả 2 zip đã kiểm tra xong
shutil.rmtree(ROOT)
for d in ["computer-vision-underwater-image-enhancer", "computer-vision-SCNet", "computer-vision-Semi-UIR"]:
    shutil.rmtree(f"/kaggle/working/{d}", ignore_errors=True)   # repo + trọng số, không cần trong output
!ls -lh /kaggle/working